In [1]:
#%pip install openpyxl

In [2]:
# Uncomment and run the following if you need to install required packages:
# %pip install torch matplotlib numpy pandas ipywidgets voila

import math
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#121212'
plt.rcParams['axes.facecolor'] = '#121212'
plt.rcParams['savefig.facecolor'] = '#121212'
plt.rcParams['text.color'] = '#f0f0f0'
plt.rcParams['axes.labelcolor'] = '#f0f0f0'
plt.rcParams['xtick.color'] = '#f0f0f0'
plt.rcParams['ytick.color'] = '#f0f0f0'
plt.rcParams['axes.edgecolor'] = '#444444'

In [3]:
# Materials and Scenario Definitions
MATERIALS = [
    "Lithium", "Cobalt", "Nickel", "Copper", "Zinc",
    "Aluminium", "Graphite", "Rare Earth Mix"
]
SCENARIOS = [
    "Balanced Data",
    "Temporal Dominant",
    "Structural Dominant",
    "Demand Shock",
    "Supply Disruption",
    "Price Surge"
]

# Synthetic demand dataset generation
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def generate_material_series(material, periods=20):
    base = 20 + 5 * MATERIALS.index(material)
    trend = np.linspace(0, 8, periods)
    season = 5 * np.sin(np.linspace(0, 4 * np.pi, periods) + MATERIALS.index(material))
    noise = np.random.normal(scale=2.5, size=periods)
    return np.clip(base + trend + season + noise, 1, None)

material_data = {
    material: generate_material_series(material)
    for material in MATERIALS
}

index = pd.date_range(start="2025-01-01", periods=20, freq='W')
demand_df = pd.DataFrame(material_data, index=index)

# Graph dependency data
adjacency = np.random.rand(len(MATERIALS), len(MATERIALS))
adjacency = (adjacency + adjacency.T) / 2
np.fill_diagonal(adjacency, 1.0)

material_embeddings = torch.randn(len(MATERIALS), 8)

# Dataset helper functions

def get_material_data(material):
    if material not in MATERIALS:
        material = MATERIALS[0]
    series = demand_df[material].values.astype(np.float32).copy()
    nodes = material_embeddings.detach().clone().float()
    return series, nodes, torch.tensor(adjacency, dtype=torch.float32)


def apply_scenario_shock(series, scenario, shock_strength):
    series = series.copy()
    if scenario == "Demand Shock":
        series[-3:] *= (1 + shock_strength)
    elif scenario == "Supply Disruption":
        series[-3:] *= (1 - shock_strength * 0.7)
    return np.clip(series, 0.5, None)


def make_scenario_title(scenario, shock_strength):
    if scenario == "Demand Shock":
        return f"Demand Shock ({shock_strength*100:.0f}% increase)"
    if scenario == "Supply Disruption":
        return f"Supply Disruption ({shock_strength*100:.0f}% effect)"
    if scenario == "Price Surge":
        return f"Price Surge ({shock_strength*100:.0f}% increase)"
    return scenario

In [4]:
# Simplified spectrally inspired TFT/ GAT models for demo purposes

class TFTDemo(nn.Module):
    def __init__(self, input_size=20, hidden_size=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, series):
        return self.net(series)

    def predict(self, series):
        with torch.no_grad():
            return self.forward(series.unsqueeze(0)).squeeze(0)

    def fit(self, x, y, epochs=60, lr=0.005):
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        loss_fn = nn.MSELoss()
        for _ in range(epochs):
            optimizer.zero_grad()
            preds = self(x)
            loss = loss_fn(preds, y)
            loss.backward()
            optimizer.step()


class GATDemo(nn.Module):
    def __init__(self, node_dim=8, hidden_size=32):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(node_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, nodes, adj):
        node_scores = self.proj(nodes).squeeze(-1)
        message = torch.matmul(adj, node_scores.unsqueeze(-1)).squeeze(-1)
        return message.mean()

    def predict(self, nodes, adj):
        with torch.no_grad():
            return self.forward(nodes, adj)

    def fit(self, nodes, adj, target, epochs=40, lr=0.005):
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        loss_fn = nn.MSELoss()
        for _ in range(epochs):
            optimizer.zero_grad()
            pred = self(nodes, adj)
            loss = loss_fn(pred.unsqueeze(0), target)
            loss.backward()
            optimizer.step()


class FusionDemo:
    def __init__(self):
        pass

    def fuse(self, tft_value, gat_value, alpha):
        return alpha * tft_value + (1 - alpha) * gat_value

    def interpret(self, alpha):
        if alpha > 0.7:
            return "Temporal-dominant fusion"
        if alpha < 0.3:
            return "Structural-dominant fusion"
        return "Balanced fusion"


class RiskDemo:
    def __call__(self, forecast, price, stock, lead_time, attn):
        budget = float((forecast * price) / 1000)
        stock_risk = float((forecast * lead_time) / max(stock, 1))
        dependency = float(attn)
        return {
            'budget_risk': budget,
            'stock_risk': stock_risk,
            'dependency_risk': dependency
        }


# Metrics

def mae(predictions, actuals):
    return float(torch.mean(torch.abs(predictions - actuals)).item())


def rmse(predictions, actuals):
    return float(torch.sqrt(torch.mean((predictions - actuals) ** 2)).item())


def mape(predictions, actuals):
    mask = actuals != 0
    return float(torch.mean(torch.abs((predictions[mask] - actuals[mask]) / actuals[mask])).item()) * 100


# Train demo models on synthetic data
series_matrix = torch.tensor(np.vstack([generate_material_series(m) for m in MATERIALS]), dtype=torch.float32)
next_targets = torch.tensor([[generate_material_series(m, periods=21)[-1]] for m in MATERIALS], dtype=torch.float32)

# Use the last 20 values per material as the training input and the next point as target
model_tft = TFTDemo(input_size=20)
model_gat = GATDemo(node_dim=8)
model_fusion = FusionDemo()
model_risk = RiskDemo()

model_tft.fit(series_matrix, next_targets)
model_gat.fit(material_embeddings, torch.tensor(adjacency, dtype=torch.float32), next_targets.mean(dim=1))

training_summary = {
    'TFT': 'trivially trained on synthetic material demand',
    'GAT': 'trivially trained on synthetic graph dependencies'
}

C:\Users\viraj.nandalikar\AppData\Roaming\Python\Python314\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([8])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [5]:
# Visualization helpers

def display_theme_css(dark_mode=True):
    background = '#121212' if dark_mode else '#f7f7f7'
    text_color = '#f5f5f5' if dark_mode else '#222222'
    widget_bg = '#1c1c1c' if dark_mode else '#ffffff'
    border_color = '#333333' if dark_mode else '#dddddd'
    display(HTML("""
<style>
body, .output {
    background-color: #121212 !important;
    color: #f5f5f5 !important;
}
.jp-Notebook {
    background: #121212 !important;
}
</style>
"""))


def plot_radar(ax, risks):
    categories = ["Budget", "Stock", "Dependency"]
    values = [risks['budget_risk'], risks['stock_risk'], risks['dependency_risk']]
    values += values[:1]
    angles = np.linspace(0, 2 * np.pi, len(categories) + 1)
    ax.set_facecolor('#121212')
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_thetagrids(angles[:-1] * (180 / np.pi), categories)
    ax.plot(angles, values, linewidth=2, linestyle='solid', color='#7c4dff')
    ax.fill(angles, values, alpha=0.25, color='#7c4dff')
    ax.set_title('Risk Radar', color='#f0f0f0')
    ax.tick_params(colors='#bbbbbb')


def plot_risk_meter(ax, risk_value, title='Overall Risk'):
    r = max(0, min(1, risk_value / max(1, risk_value)))
    ax.set_facecolor('#121212')
    ax.axis('off')
    theta = np.linspace(-0.75*np.pi, -0.25*np.pi, 100)
    ax.plot(np.cos(theta), np.sin(theta), color='#444444', linewidth=18)
    color = '#66ff66' if r < 0.4 else '#ffd54f' if r < 0.7 else '#ff5252'
    theta_indicator = np.linspace(-0.75*np.pi, -0.75*np.pi + 0.5*np.pi*r, 100)
    ax.plot(np.cos(theta_indicator), np.sin(theta_indicator), color=color, linewidth=18)
    angle = -0.75*np.pi + r * 0.5*np.pi
    ax.plot([0, np.cos(angle)], [0, np.sin(angle)], color='#ffffff', linewidth=3)
    ax.text(0, -0.2, f"{risk_value:.2f}", fontsize=14, ha='center', color='#ffffff')


def explain_model_weights(model, material_series):
    if isinstance(model, TFTDemo):
        weights = model.net[0].weight.data.abs().mean(dim=0).cpu().numpy()
        top = np.argsort(-weights)[:5]
        feature_names = [f'week_{i+1}' for i in range(len(weights))]
        return pd.DataFrame({
            'feature': [feature_names[i] for i in top],
            'importance': weights[top]
        })
    return pd.DataFrame({'feature': ['graph_dependency'], 'importance': [1.0]})


def format_metrics(name, preds, actual):
    return {
        'model': name,
        'MAE': mae(preds, actual),
        'RMSE': rmse(preds, actual),
        'MAPE': mape(preds, actual)
    }


def save_report(material, scenario, df_out):
    out_dir = Path('atsf_exports')
    out_dir.mkdir(exist_ok=True)
    path = out_dir / f'atsf_report_{material}_{scenario.replace(" ", "_")}.csv'
    df_out.to_csv(path, index=False)
    return path

In [6]:
material_dd = widgets.Dropdown(
    options=MATERIALS,
    value='Lithium',
    description='Material:',
    style={'description_width': '120px'}
)

scenario_dd = widgets.Dropdown(
    options=SCENARIOS,
    value='Balanced Data',
    description='Scenario:',
    style={'description_width': '120px'}
)

export_button = widgets.Button(
    description='Export Full Data',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

export_output = widgets.Output()
from IPython.display import FileLink
import re

latest_export = None

def clean_file_name(value):
    return re.sub(r'[^A-Za-z0-9_-]+', '_', str(value)).strip('_')


def export_full_data(b):
    global latest_export

    with export_output:
        export_output.clear_output()

        if latest_export is None:
            print("Please run the dashboard once before exporting.")
            return

        out_dir = Path("atsf_exports")
        out_dir.mkdir(exist_ok=True)

        material = clean_file_name(latest_export["material"])
        scenario = clean_file_name(latest_export["scenario"])

        file_path = out_dir / f"atsf_export_{material}_{scenario}.xlsx"

        with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
            latest_export["scenario_comparison"].to_excel(writer, sheet_name="Scenario Comparison", index=False)
            latest_export["decision_report"].to_excel(writer, sheet_name="Decision Report", index=False)
            latest_export["demand_data"].to_excel(writer, sheet_name="Demand Trend", index=False)
            latest_export["metrics"].to_excel(writer, sheet_name="Model Metrics", index=False)
            latest_export["risk"].to_excel(writer, sheet_name="Risk Scores", index=False)
            latest_export["feature_importance"].to_excel(writer, sheet_name="Feature Importance", index=False)
            latest_export["summary"].to_excel(writer, sheet_name="Business Summary", index=False)

        print(f"Export completed: {file_path}")
        display(FileLink(file_path))

# ✅ Inline styled labels
material_row = widgets.HBox([
    widgets.HTML("<div style='background:#2a5298; color:white; padding:4px 10px; border-radius:6px;'>Material</div>"),
    material_dd
])

scenario_row = widgets.HBox([
    widgets.HTML("<div style='background:#2a5298; color:white; padding:4px 10px; border-radius:6px;'>Scenario</div>"),
    scenario_dd
])

ui = widgets.VBox([
    material_row,
    scenario_row,
    export_button
])

In [7]:
def interpret_graph(material, scenario, series, adjusted_series):
    text = "<h3>📘 Demand Interpretation</h3>"

    trend = "increasing" if adjusted_series[-1] > adjusted_series[0] else "decreasing"

    text += f"<p><b>Trend:</b> The demand for {material} is showing a <b>{trend}</b> pattern over time.</p>"

    if scenario == "Demand Shock":
        text += "<p>⚡ <b>Demand Shock:</b> A sharp increase is visible in recent periods, indicating sudden surge in demand.</p>"

    elif scenario == "Supply Disruption":
        text += "<p>⚠️ <b>Supply Disruption:</b> Recent decline suggests supply constraints affecting availability.</p>"

    elif scenario == "Price Surge":
        text += "<p>💰 <b>Price Surge:</b> Demand pattern may become volatile due to rising costs.</p>"

    elif scenario == "Temporal Dominant":
        text += "<p>⏳ <b>Temporal Focus:</b> Model prioritizes historical demand trends.</p>"

    elif scenario == "Structural Dominant":
        text += "<p>🔗 <b>Structural Focus:</b> Model emphasizes dependency relationships between materials.</p>"

    else:
        text += "<p>⚖️ <b>Balanced Scenario:</b> Stable demand without extreme fluctuations.</p>"

    # volatility insight
    volatility = np.std(adjusted_series)
    text += f"<p><b>Volatility:</b> Demand variability is <b>{volatility:.2f}</b>, indicating stability level.</p>"

    return HTML(f"""
    <div style='background:#1a1a1a; padding:16px; border-radius:10px; border:1px solid #333; color:#e0e0e0;'>
        {text}
    </div>
    """)

In [8]:
def create_metrics_carousel(metrics_df, risk_df, explain_df):
    carousel_id = "carousel_" + str(np.random.randint(10000))

    html = f"""
    <div id="{carousel_id}" style="width:100%; overflow:hidden;">
        <div class="carousel-track" style="display:flex; transition: transform 0.6s ease;">
            
            <div class="carousel-slide">{metrics_df.to_html(index=False)}</div>
            <div class="carousel-slide">{risk_df.to_html(index=False)}</div>
            <div class="carousel-slide">{explain_df.to_html(index=False)}</div>
            
        </div>
        
        <div style="text-align:center; margin-top:10px;">
            <button onclick="prevSlide_{carousel_id}()">⬅️</button>
            <button onclick="nextSlide_{carousel_id}()">➡️</button>
        </div>
    </div>

    <style>
        #{carousel_id} table {{
            width: 80% !important;          /* ✅ reduced width */
            margin: auto;                  /* ✅ center table */
            border-collapse: collapse;
            background:#1e1e1e !important;
            color:#f0f0f0 !important;
        }}

        #{carousel_id} th {{
            background:#7c4dff !important;
            color:white !important;
            padding:6px;
            font-size:13px;
        }}

        #{carousel_id} td {{
            padding:6px;
            border:1px solid #333;
            background:#1e1e1e !important;
            color:#f0f0f0 !important;
            font-size:13px;
        }}

        .carousel-slide {{
            min-width:100%;
            padding:10px;
            display:flex;
            justify-content:center; 
        }}
    </style>

    <script>
    let index_{carousel_id} = 0;

    function updateSlide_{carousel_id}() {{
        const track = document.querySelector("#{carousel_id} .carousel-track");
        track.style.transform = `translateX(-${{index_{carousel_id} * 100}}%)`;
    }}

    function nextSlide_{carousel_id}() {{
        index_{carousel_id} = (index_{carousel_id} + 1) % 3;
        updateSlide_{carousel_id}();
    }}

    function prevSlide_{carousel_id}() {{
        index_{carousel_id} = (index_{carousel_id} - 1 + 3) % 3;
        updateSlide_{carousel_id}();
    }}
    </script>
    """

    return HTML(html)

In [9]:
def interpret_demand(material, scenario, series):
    trend = "increasing" if series[-1] > series[0] else "decreasing"
    volatility = np.std(series)

    text = f"""
    <h3>📘 Demand Interpretation</h3>
    <p><b>Material:</b> {material}</p>
    <p><b>Trend:</b> {trend}</p>
    <p><b>Volatility:</b> {volatility:.2f}</p>
    """

    if scenario == "Demand Shock":
        text += "<p>⚡ Sudden demand spike observed.</p>"
    elif scenario == "Supply Disruption":
        text += "<p>⚠️ Drop reflects supply constraints.</p>"
    elif scenario == "Price Surge":
        text += "<p>💰 Price fluctuation affecting demand.</p>"
    elif scenario == "Temporal Dominant":
        text += "<p>⏳ Driven by historical patterns.</p>"
    elif scenario == "Structural Dominant":
        text += "<p>🔗 Influenced by dependencies.</p>"
    else:
        text += "<p>⚖️ Stable balanced demand.</p>"

    return HTML(f"""
    <div style='background:#1a1a1a; color:#e0e0e0; padding:15px; border-radius:10px; border:1px solid #333;'>
        {text}
    </div>
    """)

In [10]:
def interpret_business(material, scenario, series):
    trend = "increasing" if series[-1] > series[0] else "decreasing"
    peak = max(series)
    low = min(series)
    recent_change = series[-1] - series[-4]
    recent_direction = "increased" if recent_change > 0 else "decreased"
    volatility = np.std(series)

    if volatility < 4:
        volatility_label = "low"
    elif volatility < 8:
        volatility_label = "moderate"
    else:
        volatility_label = "high"

    explanation = f"""
    <h3 style='margin-top:0; color:#80d8ff;'>Business Interpretation</h3>

    <p>
        Demand for <b>{material}</b> is generally <b>{trend}</b> across the selected period.
    </p>

    <p>
        The highest observed demand is <b>{peak:.2f}</b>, while the lowest is <b>{low:.2f}</b>.
        This gives a quick view of the demand range the business may need to plan for.
    </p>

    <p>
        In the most recent periods, demand has <b>{recent_direction}</b> by
        <b>{abs(recent_change):.2f}</b> units.
    </p>

    <p>
        Volatility is <b>{volatility_label}</b> with a variability score of
        <b>{volatility:.2f}</b>.
    </p>
    """

    if scenario == "Demand Shock":
        explanation += """
        <p><b>Business meaning:</b> Recent demand is rising sharply. Procurement and inventory teams
        may need to prepare for faster stock movement and possible shortage risk.</p>
        """
    elif scenario == "Supply Disruption":
        explanation += """
        <p><b>Business meaning:</b> Demand or availability is weakening in recent periods.
        This may indicate supply-side pressure and a need to review supplier reliability.</p>
        """
    elif scenario == "Price Surge":
        explanation += """
        <p><b>Business meaning:</b> Demand may become less stable as price pressure increases.
        Budget planning and purchasing timing become more important.</p>
        """
    elif scenario == "Temporal Dominant":
        explanation += """
        <p><b>Business meaning:</b> The pattern is mainly explained by historical demand behavior.
        Past movement is a strong guide for the current forecast.</p>
        """
    elif scenario == "Structural Dominant":
        explanation += """
        <p><b>Business meaning:</b> Demand is strongly influenced by relationships with other materials.
        Dependency risk should be monitored along with the material's own demand.</p>
        """
    else:
        explanation += """
        <p><b>Business meaning:</b> Demand is relatively stable without a major disruption signal.
        Standard planning assumptions are likely reasonable.</p>
        """

    return f"""
    <div style='
        background:#1a1a1a;
        color:#f0f0f0;
        padding:18px;
        border-radius:10px;
        border:1px solid #333;
        height:100%;
        box-sizing:border-box;
        line-height:1.45;
        font-size:14px;
    '>
        {explanation}
    </div>
    """

In [11]:
def get_business_recommendation(forecast, risk_score, volatility, scenario):
    if scenario == "Demand Shock":
        return "Buy Now" if risk_score >= 0.6 else "Increase Safety Stock"

    if scenario == "Supply Disruption":
        return "Review Supplier" if risk_score >= 0.5 else "Monitor Supplier Capacity"

    if scenario == "Price Surge":
        return "Buy Early" if risk_score >= 0.5 else "Monitor Price Movement"

    if volatility >= 8:
        return "Monitor Closely"

    if risk_score >= 0.7:
        return "Immediate Action Required"

    if risk_score >= 0.4:
        return "Monitor"

    return "Standard Planning"


def get_risk_level(risk_score):
    if risk_score >= 0.75:
        return "Critical"
    if risk_score >= 0.55:
        return "High"
    if risk_score >= 0.35:
        return "Moderate"
    return "Low"


def build_scenario_comparison(material):
    comparison_rows = []

    base_series, nodes, adj = get_material_data(material)

    for scenario_name in SCENARIOS:
        scenario_series = apply_scenario_shock(base_series, scenario_name, 0.2)

        ts_input = torch.tensor(scenario_series[-20:], dtype=torch.float32)
        tft_value = model_tft.predict(ts_input)
        gat_value = model_gat.predict(nodes, adj)

        alpha_map = {
            "Temporal Dominant": 0.85,
            "Structural Dominant": 0.15,
            "Balanced Data": 0.5
        }
        alpha = alpha_map.get(scenario_name, 0.5)

        fused_value = model_fusion.fuse(tft_value, gat_value, alpha)

        risk_output = model_risk(
            forecast=fused_value,
            price=torch.tensor([50.0]),
            stock=torch.tensor([200.0]),
            lead_time=torch.tensor([15.0]),
            attn=torch.tensor([0.35])
        )

        overall_risk = np.mean([
            risk_output["budget_risk"],
            risk_output["stock_risk"],
            risk_output["dependency_risk"]
        ])

        volatility = float(np.std(scenario_series))

        comparison_rows.append({
            "Scenario": scenario_name,
            "Forecast": round(float(fused_value.item()), 2),
            "Avg Demand": round(float(np.mean(scenario_series)), 2),
            "Volatility": round(volatility, 2),
            "Risk Score": round(float(overall_risk), 2),
            "Risk Level": get_risk_level(overall_risk),
            "Recommendation": get_business_recommendation(
                float(fused_value.item()),
                overall_risk,
                volatility,
                scenario_name
            )
        })

    return pd.DataFrame(comparison_rows)

In [12]:
def build_business_rules_section():
    rules_html = """
    <div style='
        background:#1a1a1a;
        color:#f0f0f0;
        padding:18px;
        border-radius:10px;
        border:1px solid #333;
        line-height:1.5;
        margin-top:16px;
    '>
        <h3 style='margin-top:0; color:#80d8ff;'>Business Rules Used</h3>

        <p><b>Risk Level:</b> Based on the average of budget risk, stock risk, and dependency risk.</p>

        <p><b>Critical Risk:</b> Risk score is 0.75 or higher. Immediate procurement or supplier action is required.</p>

        <p><b>High Risk:</b> Risk score is between 0.55 and 0.75. Business teams should review inventory, supplier exposure, and budget impact.</p>

        <p><b>Moderate Risk:</b> Risk score is between 0.35 and 0.55. The material should be monitored, but urgent action may not be required.</p>

        <p><b>Low Risk:</b> Risk score is below 0.35. Standard planning assumptions are acceptable.</p>

        <p><b>Demand Shock Rule:</b> If demand rises sharply, the recommendation shifts toward buying now or increasing safety stock.</p>

        <p><b>Supply Disruption Rule:</b> If supply disruption is selected, supplier review becomes the priority.</p>

        <p><b>Price Surge Rule:</b> If price pressure is selected, early purchasing or price monitoring becomes more important.</p>

        <p><b>Volatility Rule:</b> If volatility is high, the system recommends closer monitoring even when the overall risk score is not critical.</p>
    </div>
    """

    return HTML(rules_html)

In [13]:
def atsf_run(material, scenario):
    clear_output(wait=True)

    series, nodes, adj = get_material_data(material)
    actual_series = apply_scenario_shock(series, scenario, 0.2)

    actual_next = torch.tensor([actual_series[-1]], dtype=torch.float32)
    ts_input = torch.tensor(actual_series[-20:], dtype=torch.float32)

    # Predictions
    tft_pred = model_tft.predict(ts_input)
    gat_pred = model_gat.predict(nodes, adj)

    # Alpha logic
    alpha_map = {
        'Temporal Dominant': 0.85,
        'Structural Dominant': 0.15,
        'Balanced Data': 0.5
    }
    current_alpha = alpha_map.get(scenario, 0.5)

    fused_pred = model_fusion.fuse(tft_pred, gat_pred, current_alpha)

    # Risk
    risk_output = model_risk(
        forecast=fused_pred,
        price=torch.tensor([50.0]),
        stock=torch.tensor([200.0]),
        lead_time=torch.tensor([15.0]),
        attn=torch.tensor([0.35])
    )

    # Metrics
    metrics = pd.DataFrame([
        format_metrics('TFT', tft_pred, actual_next),
        format_metrics('GAT', gat_pred.unsqueeze(0), actual_next),
        format_metrics('ATSF', fused_pred.unsqueeze(0), actual_next)
    ])

    weight_df = explain_model_weights(model_tft, ts_input)

    risk_df = pd.DataFrame([risk_output])
    scenario_comparison_df = build_scenario_comparison(material)

    overall_risk_score = float(np.mean([
    risk_output["budget_risk"],
    risk_output["stock_risk"],
    risk_output["dependency_risk"]
    ]))

    current_volatility = float(np.std(actual_series))

    decision_report_df = pd.DataFrame([{
        "Material": material,
        "Selected Scenario": scenario,
        "ATSF Forecast": round(float(fused_pred.item()), 2),
        "Average Demand": round(float(np.mean(actual_series)), 2),
        "Lowest Demand": round(float(np.min(actual_series)), 2),
        "Highest Demand": round(float(np.max(actual_series)), 2),
        "Volatility": round(current_volatility, 2),
        "Risk Score": round(overall_risk_score, 2),
        "Risk Level": get_risk_level(overall_risk_score),
        "Fusion Type": model_fusion.interpret(current_alpha),
        "Recommended Action": get_business_recommendation(
            float(fused_pred.item()),
            overall_risk_score,
            current_volatility,
            scenario
        ),
        "Business Reason": (
            "Decision is based on forecasted demand, scenario impact, volatility, "
            "stock risk, budget risk, and dependency risk."
        )
    }])

    display(build_business_rules_section())

    global latest_export

    demand_export_df = pd.DataFrame({
        "date": demand_df.index.strftime("%Y-%m-%d"),
        "material": material,
        "scenario": scenario,
        "historical_demand": demand_df[material].values,
        "scenario_adjusted_demand": actual_series
    })

    summary_df = pd.DataFrame([{
        "material": material,
        "scenario": scenario,
        "trend": "increasing" if actual_series[-1] > actual_series[0] else "decreasing",
        "lowest_demand": float(np.min(actual_series)),
        "highest_demand": float(np.max(actual_series)),
        "average_demand": float(np.mean(actual_series)),
        "volatility": float(np.std(actual_series)),
        "tft_forecast": float(tft_pred.item()),
        "gat_forecast": float(gat_pred.item()),
        "atsf_forecast": float(fused_pred.item()),
        "fusion_type": model_fusion.interpret(current_alpha)
    }])

    latest_export = {
        "material": material,
        "scenario": scenario,
        "demand_data": demand_export_df,
        "metrics": metrics,
        "risk": risk_df,
        "feature_importance": weight_df,
        "summary": summary_df,
        "scenario_comparison": scenario_comparison_df,
        "decision_report": decision_report_df
    }

    display(HTML("<h3 style='color:#80d8ff;'>Insights</h3>"))
    display(create_metrics_carousel(metrics, risk_df, weight_df))
    graph_output = widgets.Output()

    with graph_output:
        fig = plt.figure(figsize=(8, 5))
        fig.patch.set_facecolor('#121212')

        import matplotlib.dates as mdates
        dates = demand_df.index

        ax1 = fig.add_subplot(1, 1, 1)
        ax1.set_facecolor('#121212')

        ax1.plot(dates, demand_df[material], color='#82cceb', marker='o', label='Historical')
        ax1.plot(
            dates[-len(actual_series):],
            actual_series,
            color='#ffa726',
            linestyle='--',
            marker='o',
            label='Scenario'
        )

        ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
        ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

        for label in ax1.get_xticklabels():
            label.set_rotation(30)
            label.set_color('white')

        ax1.set_title(f'Demand Trend ({material})', color='white')
        ax1.set_xlabel('Date', color='white')
        ax1.set_ylabel('Demand', color='white')
        ax1.tick_params(colors='white')
        ax1.legend()

        plt.tight_layout()
        plt.show()

    business_panel = widgets.HTML(
        value=interpret_business(material, scenario, actual_series),
        layout=widgets.Layout(width='36%', min_width='320px')
    )

    display(
        widgets.HBox(
            [
                widgets.Box(
                    [graph_output],
                    layout=widgets.Layout(width='64%')
                ),
                business_panel
            ],
            layout=widgets.Layout(
                width='100%',
                align_items='stretch',
                gap='18px'
            )
        )
    )

    display(HTML("<h3 style='color:#80d8ff;'>Scenario Comparison Table</h3>"))
    display(HTML(
        scenario_comparison_df.to_html(
            index=False,
            classes="scenario-table"
        )
    ))


In [14]:
import subprocess
import sys
import time
import webbrowser
from pathlib import Path

notebook_path = Path("ATSF_Dashboard_Extended.ipynb")

launch_button = widgets.Button(
    description="🚀 Launch Dashboard Extended",
    button_style='success',
    layout=widgets.Layout(width='300px', height='40px')
)

launch_output = widgets.Output()

PORT = 8867

def launch_voila(b):
    with launch_output:
        launch_output.clear_output()

        try:
            subprocess.run(
                [sys.executable, '-m', 'voila', '--version'],
                capture_output=True, check=True, text=True
            )
        except Exception:
            print("Voila not installed. Run: pip install voila")
            return

        print(f"Starting Voila on port {PORT}...")

        try:
            # ✅ IMPORTANT: specify port
            subprocess.Popen(
                [
                    sys.executable, '-m', 'voila',
                    str(notebook_path),
                    '--no-browser',
                    f'--port={PORT}'
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE
            )

            time.sleep(2)

            webbrowser.open(f'http://localhost:{PORT}')
            print(f"Opened at http://localhost:{PORT}")

        except Exception as e:
            print("Error launching Voila:", e)
            
out = widgets.interactive_output(
    atsf_run,
    {
        'material': material_dd,
        'scenario': scenario_dd
    }
)
launch_button.on_click(launch_voila)
panel = widgets.VBox([
    widgets.HTML("<h2 style='color:#7c4dff;'>ATSF Dashboard Extended</h2>"),
    ui,
    export_output,
    out
])
display(HTML("""
<style>
body, .output {
    background-color: #121212 !important;
    color: #f5f5f5 !important;
}
.jp-Notebook, .jp-Cell {
    background-color: #121212 !important;
}
</style>
"""))

display(HTML("""
<style>
.scenario-table {
    width: 100%;
    border-collapse: collapse;
    background: #1e1e1e;
    color: #f0f0f0;
    font-size: 13px;
}

.scenario-table th {
    background: #2a5298;
    color: white;
    padding: 8px;
    text-align: left;
}

.scenario-table td {
    border: 1px solid #333;
    padding: 8px;
}

.scenario-table tr:nth-child(even) {
    background: #181818;
}
</style>
"""))

export_button.on_click(export_full_data)
display(panel)
display(widgets.VBox([launch_button, launch_output]))